# HARs Burden Test

## Install RVTests


In [ ]:
%%bash
if test -e /home/jupyter/tools/rvtests/executable/rvtest; then
    echo "RVtests is already installed:"
    ls -la /home/jupyter/tools/rvtests/executable/rvtest
else
    echo "Installing RVtests..."
    mkdir -p /home/jupyter/tools/rvtests
    cd /home/jupyter/tools/rvtests
    wget -q https://github.com/zhanxw/rvtests/releases/download/v2.1.0/rvtests_linux64.tar.gz
    tar -zxvf rvtests_linux64.tar.gz
    chmod 777 /home/jupyter/tools/rvtests/executable/rvtest
    echo ""
    echo "Done:"
    ls -la /home/jupyter/tools/rvtests/executable/rvtest
fi

## Setup, imports and configuration

In [ ]:
import os
import csv
import time
import pathlib
import subprocess
from datetime import datetime
from multiprocessing import Manager
from concurrent.futures import ProcessPoolExecutor, as_completed

import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# Configuration
release = 12

ANCESTRIES = ['AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH']
# Some ancestries are expected to be underpowered (small N and/or extreme case-control
# imbalance); they're still run, just not treated as standalone findings (see the note
# near the final summary).

DATASETS   = ['NBA', 'WGS']

# Burden classes
BURDEN_CLASSES = {
    "maf01": 0.01,    # rare variants
    "maf03": 0.03,    # less rare (sensitivity analysis)
}

# Covariates: PC1-PC5
COVAR_NAME = "SEX,AGE,PC1,PC2,PC3,PC4,PC5"

MAX_WORKERS = 16

# Paths
DIR_TOOL  = "/home/jupyter/tools"
DIR_WSPS  = "/home/jupyter/workspace/ws_files"
DIR_NOVA  = f"{DIR_WSPS}/Novalis_v3_R12"
DIR_HARS_REF = f"{DIR_WSPS}/HARS_files/HARs_merged"
DIR_BURDEN = f"{DIR_NOVA}/Burden_union"
DIR_RESU   = f"{DIR_NOVA}/Results_union"

RVTEST  = f"{DIR_TOOL}/rvtests/executable/rvtest"

# Deduplicated (union) HAR list - consolidated canonical location
HAR_LIST_FILE = f"{DIR_HARS_REF}/HAR_list_phase_1_union.tsv"

# Custom refFlat for the union HARs - rebuilt under Novalis_v3_R12 (a derived
# build artifact, doesn't touch the canonical one)
HAR_REFFLAT = f"{DIR_NOVA}/HAR_refFlat_union_hg38.txt"

# Create base directories
for d in [DIR_NOVA, DIR_BURDEN, DIR_RESU]:
    os.makedirs(d, exist_ok=True)
for ds in DATASETS:
    for anc in ANCESTRIES:
        os.makedirs(f"{DIR_BURDEN}/{ds}/{anc}", exist_ok=True)

print(f"Ancestries:   {ANCESTRIES}")
print(f"Datasets:     {DATASETS}")
print(f"Burden classes: {list(BURDEN_CLASSES.keys())}")
print(f"Covar-name:   {COVAR_NAME}")
print(f"MAX_WORKERS:  {MAX_WORKERS}")
print(f"HAR list:     {HAR_LIST_FILE}")
print(f"Output root:  {DIR_NOVA}")
print(f"Burden dir:   {DIR_BURDEN}")
print(f"Results dir:  {DIR_RESU}")

In [ ]:
# Check samplestokeep.txt in Novalis_v3_R12
import pandas as pd

SAMPLESTOKEEP = f"{DIR_NOVA}/samplestokeep.txt"

if not os.path.isfile(SAMPLESTOKEEP):
    raise FileNotFoundError(
        f"{SAMPLESTOKEEP} not found.\n"
        "The R12 covariate builder should have generated it automatically under Novalis_v3_R12/."
    )

stk = pd.read_csv(SAMPLESTOKEEP, sep='\t', header=None, names=['GP2ID', 'PHENO'])
n_total = len(stk)
n_pd    = (stk['PHENO'] == 2).sum()
n_ctrl  = (stk['PHENO'] == 1).sum()
n_other = n_total - n_pd - n_ctrl

print(f"samplestokeep.txt (R12): {SAMPLESTOKEEP}")
print(f"  N total: {n_total:,}  |  PD (2): {n_pd:,}  |  Control (1): {n_ctrl:,}  |  other values: {n_other:,}")

if n_other > 0:
    print("  Found PHENO values other than 1/2 - check the file.")
else:
    print("  PHENO only contains 1 (Control) and 2 (PD).")

# Guard rail: existing covariate files must be a subset of samplestokeep
valid_ids = set(stk['GP2ID'].astype(str))
print("\nChecking covariate files (Working_{ds}_v3):")
any_covariate_files = False
for ds in DATASETS:
    for anc in ANCESTRIES:
        cov = f"{DIR_WSPS}/Working_{ds}_v3/{anc}/InputFiles/{anc}_covariate_file.txt"
        if os.path.isfile(cov):
            any_covariate_files = True
            try:
                cdf = pd.read_csv(cov, sep=None, engine='python')
                ids_cov = set(cdf['IID'].astype(str))
                not_in_master = ids_cov - valid_ids
                if not_in_master:
                    print(f"  {ds} {anc}: {len(not_in_master)} IDs NOT in samplestokeep")
                else:
                    print(f"  {ds} {anc}: {len(ids_cov)} IDs OK")
            except Exception as e:
                print(f"  ? {ds} {anc}: could not check ({e})")
if not any_covariate_files:
    print("  (No covariate files in Working_{ds}_v3 - run the R12 covariate builder first)")

## Load HAR list

In [ ]:
with open(HAR_LIST_FILE, 'r') as f:
    reader = csv.reader(f, delimiter='\t')
    HARS_DICT = {
        row[3]: {
            'name':  row[3],
            'chrom': row[0].replace('chr', ''),
            'start': int(row[1]),
            'end':   int(row[2]),
        }
        for row in reader
    }

print(f"HARs loaded: {len(HARS_DICT)}")
print(f"Example: {next(iter(HARS_DICT.items()))}")

# Bonferroni threshold for reporting (per-ancestry only; cross-ancestry reporting was dropped)
BONF_PER_ANC = 0.05 / len(HARS_DICT)
print(f"\nBonferroni per-ancestry: alpha = 0.05 / {len(HARS_DICT)} = {BONF_PER_ANC:.3e}")

## Build a custom refFlat for the union HAR list


In [ ]:
with open(HAR_REFFLAT, 'w') as f:
    for HAR, info in HARS_DICT.items():
        chrom = info['chrom'] if info['chrom'].startswith('chr') else f'chr{info["chrom"]}'
        start = info['start']
        end   = info['end']
        f.write('\t'.join([
            HAR, HAR, chrom, '+',
            str(start), str(end),
            str(start), str(end),
            '1',
            f'{start},',
            f'{end},'
        ]) + '\n')

print(f"HAR refFlat written: {HAR_REFFLAT}")
!head -3 {HAR_REFFLAT}
!wc -l {HAR_REFFLAT}

## Check inputs and build `present_hars` per (dataset, ancestry)


In [ ]:
def vcf_path(ds, anc, HAR):
    return f"{DIR_WSPS}/results_region_extrac_v3_{ds}/{anc}/InputFiles/Indiv_HARS/{HAR}.vcf.gz"

def covar_path(ds, anc):
    # Covariates stay under the original Working_{ds}_v3/ (not _union)
    return f"{DIR_WSPS}/Working_{ds}_v3/{anc}/InputFiles/{anc}_covariate_file.txt"

summary = {}
present_hars = {}    # present_hars[ds][anc] = list of HARs

for ds in DATASETS:
    present_hars[ds] = {}
    summary[ds] = {}
    for anc in ANCESTRIES:
        cov = covar_path(ds, anc)
        cov_ok = os.path.isfile(cov)

        indiv_dir = f"{DIR_WSPS}/results_region_extrac_v3_{ds}/{anc}/InputFiles/Indiv_HARS"
        try:
            files = set(os.listdir(indiv_dir))
        except FileNotFoundError:
            files = set()

        present = [HAR for HAR in HARS_DICT
                   if f"{HAR}.vcf.gz" in files and f"{HAR}.vcf.gz.tbi" in files]

        present_hars[ds][anc] = present

        summary[ds][anc] = {
            'covariate_ok':   cov_ok,
            'n_vcfs_present': len(present),
            'n_missing':      len(HARS_DICT) - len(present),
        }

# Print summary
print(f"{'DS':<6} {'ANC':<5} {'COVAR_OK':<9} {'PRESENT':>8} {'MISSING':>8} {'%COVERAGE':>11}")
print("-" * 50)
for ds in DATASETS:
    for anc in ANCESTRIES:
        s = summary[ds][anc]
        pct = 100 * s['n_vcfs_present'] / len(HARS_DICT)
        ok = 'YES' if s['covariate_ok'] else 'NO'
        print(f"{ds:<6} {anc:<5} {ok:<9} {s['n_vcfs_present']:>8} {s['n_missing']:>8} {pct:>10.1f}%")

## Burden test function (with resume + OOM retry)

- **Resume**: if both `.assoc` files already exist and aren't empty, marks it `[CACHED]` and skips.
- **Retry on SIGKILL**: if rvtest dies with `rc=-9` (OOM killer), waits 5s and retries once.

In [ ]:
def burden_rvtest(ds, anc, HAR, burden_class, freq_upper):
    """Run RVTests SKAT/SKAT-O with resume and SIGKILL retry."""
    in_vcf = vcf_path(ds, anc, HAR)
    covar  = covar_path(ds, anc)

    if not os.path.isfile(in_vcf):
        return f"[SKIP] {ds} {anc} {HAR} {burden_class} - no input VCF"
    if not os.path.isfile(covar):
        return f"[SKIP] {ds} {anc} {HAR} {burden_class} - no covariate file"

    out_dir = f"{DIR_BURDEN}/{ds}/{anc}"
    os.makedirs(out_dir, exist_ok=True)
    out_prefix = f"{out_dir}/{HAR}_{burden_class}.burden"

    # Resume: skip if both .assoc files already exist and aren't empty
    skat_done  = os.path.isfile(f"{out_prefix}.Skat.assoc")  and os.path.getsize(f"{out_prefix}.Skat.assoc")  > 0
    skato_done = os.path.isfile(f"{out_prefix}.SkatO.assoc") and os.path.getsize(f"{out_prefix}.SkatO.assoc") > 0
    if skat_done and skato_done:
        return f"[CACHED] {ds} {anc} {HAR} {burden_class}"

    cmd = [
        RVTEST,
        "--noweb", "--hide-covar",
        "--inVcf",     in_vcf,
        "--out",       out_prefix,
        "--kernel",    "skat,skato",
        "--pheno",     covar,
        "--pheno-name","PHENO",
        "--covar",     covar,
        "--covar-name",COVAR_NAME,
        "--gene",      HAR,
        "--geneFile",  HAR_REFFLAT,
    ]
    if freq_upper is not None:
        cmd += ["--freqUpper", str(freq_upper)]

    # Retry: up to 2 attempts if rc=-9 (SIGKILL/OOM)
    for attempt in (1, 2):
        ts = datetime.now().strftime('%H:%M:%S')
        try:
            result = subprocess.run(cmd, capture_output=True, text=True, check=False)
            if result.returncode == 0:
                tag = " (retry)" if attempt == 2 else ""
                return f"[DONE] {ts} {ds} {anc} {HAR} {burden_class}{tag}"
            if result.returncode == -9 and attempt == 1:
                time.sleep(5)
                continue
            return f"[FAIL] {ts} {ds} {anc} {HAR} {burden_class} | rc={result.returncode} | {result.stderr[-150:]}"
        except Exception as e:
            return f"[ERR ] {ts} {ds} {anc} {HAR} {burden_class} | {e}"
    return f"[FAIL2] {ts} {ds} {anc} {HAR} {burden_class} | OOM x2"


def _run_task(args):
    return burden_rvtest(*args)

## Run burden tests across all ancestries (parallel)


In [ ]:
# Build task list: (ds, anc, HAR, class, freqUpper)
tasks = [
    (ds, anc, HAR, cls_name, cls_freq)
    for ds in DATASETS
    for anc in ANCESTRIES
    for HAR in present_hars[ds][anc]
    for cls_name, cls_freq in BURDEN_CLASSES.items()
]

print(f"Total tasks: {len(tasks):,}")
for ds in DATASETS:
    sub_n = sum(len(present_hars[ds][anc]) for anc in ANCESTRIES) * len(BURDEN_CLASSES)
    print(f"  {ds}: {sub_n:,} tasks")
print(f"Workers: {MAX_WORKERS}\n")

ts_start = datetime.now()
with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = {ex.submit(_run_task, t): t for t in tasks}
    completed = 0
    n_done = n_cached = n_fail = n_skip = 0
    for fut in as_completed(futures):
        completed += 1
        try:
            msg = fut.result()
        except Exception as e:
            msg = f"[EXC ] {futures[fut]} | {e}"

        if   msg.startswith('[DONE'):   n_done   += 1
        elif msg.startswith('[CACHED'): n_cached += 1
        elif msg.startswith('[SKIP'):   n_skip   += 1
        else:                            n_fail   += 1

        if completed % 500 == 0 or msg.startswith(('[FAIL', '[ERR ', '[EXC ')):
            elapsed = datetime.now() - ts_start
            rate = completed / max(elapsed.total_seconds() / 60, 0.01)
            eta_min = (len(tasks) - completed) / max(rate, 0.01)
            print(f"[{completed:,}/{len(tasks):,}] D={n_done} C={n_cached} F={n_fail} S={n_skip} | {rate:.1f}/min | ETA {eta_min/60:.1f}h | {msg}")

print(f"\nFinished in {datetime.now() - ts_start}")
print(f"Summary: DONE={n_done:,}  CACHED={n_cached:,}  FAIL={n_fail:,}  SKIP={n_skip:,}")

## Compile results per ancestry (SKAT and SKAT-O)

Collects every `.assoc` file into per-ancestry/dataset TSVs, adding `ANCESTRY, GENE, CLASS, DATASET` columns.

In [ ]:
# Self-contained compilation cell (doesn't need earlier cells to have run)
import os, csv
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

DIR_WSPS   = "/home/jupyter/workspace/ws_files"
DIR_BURDEN = f"{DIR_WSPS}/Novalis_v3_R12/Burden_union"
DIR_RESU   = f"{DIR_WSPS}/Novalis_v3_R12/Results_union"
HAR_LIST_FILE   = f"{DIR_WSPS}/HARS_files/HARs_merged/HAR_list_phase_1_union.tsv"
ANCESTRIES = ['AAC','AFR','AJ','AMR','CAS','EAS','EUR','FIN','MDE','SAS','CAH']
DATASETS   = ['NBA','WGS']
CLASSES    = ['maf01','maf03']
N_WORKERS  = 32
os.makedirs(DIR_RESU, exist_ok=True)

with open(HAR_LIST_FILE) as f:
    hars_all = [row[3] for row in csv.reader(f, delimiter='\t')]

# Rebuild present_hars from disk
present_hars = {}
for ds in DATASETS:
    present_hars[ds] = {}
    for anc in ANCESTRIES:
        indiv_dir = f"{DIR_WSPS}/results_region_extrac_v3_{ds}/{anc}/InputFiles/Indiv_HARS"
        try:
            vcf_files = set(os.listdir(indiv_dir))
            present_hars[ds][anc] = [h for h in hars_all if f"{h}.vcf.gz" in vcf_files]
        except FileNotFoundError:
            present_hars[ds][anc] = []

def read_assoc_fast(path):
    try:
        if not os.path.isfile(path) or os.path.getsize(path) == 0:
            return None
        with open(path) as f:
            rows = list(csv.DictReader(f, delimiter='\t'))
        return rows[0] if rows else None
    except Exception:
        return None

def compile_one(ds, anc, kernel_suffix, hars, classes):
    burden_dir = Path(f"{DIR_BURDEN}/{ds}/{anc}")
    paths, metas = [], []
    for har in hars:
        for cls in classes:
            paths.append(str(burden_dir / f"{har}_{cls}.burden.{kernel_suffix}.assoc"))
            metas.append((har, cls))
    rows = []
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futures = {ex.submit(read_assoc_fast, p): m for p, m in zip(paths, metas)}
        for fut in as_completed(futures):
            har, cls = futures[fut]
            row = fut.result()
            if row is not None:
                row.update({'ANCESTRY': anc, 'GENE': har, 'CLASS': cls, 'DATASET': ds})
                rows.append(row)
    return pd.DataFrame(rows)

# Compile
ts_start = datetime.now()
for ds in DATASETS:
    for anc in ANCESTRIES:
        if not present_hars[ds][anc]:
            continue
        for kernel_suffix, kernel_name in [('Skat','SKAT'), ('SkatO','SKAT-O')]:
            df = compile_one(ds, anc, kernel_suffix, present_hars[ds][anc], CLASSES)
            if not df.empty:
                front = ['ANCESTRY','GENE','CLASS','DATASET']
                df = df[front + [c for c in df.columns if c not in front]]
            out = f"{DIR_RESU}/{ds}_{anc}_BURDEN.{kernel_name}.tsv"
            df.to_csv(out, sep='\t', index=False)
            print(f"[{datetime.now():%H:%M:%S}] {ds} {anc:<5} {kernel_name:<7}: {len(df):>5} rows -> {out.split('/')[-1]}")

print(f"\nFinished in {datetime.now() - ts_start}")

## Bonferroni and FDR-BH correction


In [ ]:
# FDR-BH and Bonferroni correction
import os
import pandas as pd
import numpy as np
from statsmodels.stats.multitest import multipletests
from datetime import datetime

DIR_RESU = "/home/jupyter/workspace/ws_files/Novalis_v3_R12/Results_union"
ANCESTRIES = ['AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH']
DATASETS   = ['NBA', 'WGS']

# Per-ancestry Bonferroni threshold (based on the actual number of tests);
# the cross-ancestry threshold was dropped.
N_HARS_UNION = 5915
BONF_PER_ANC = 0.05 / N_HARS_UNION   # 8.45e-6

print(f"{datetime.now().strftime('%H:%M:%S')}  Loading TSVs...")

# Load every per-ancestry/dataset TSV
all_skat  = []
all_skato = []

for ds in DATASETS:
    for anc in ANCESTRIES:
        path_skat  = f"{DIR_RESU}/{ds}_{anc}_BURDEN.SKAT.tsv"
        path_skato = f"{DIR_RESU}/{ds}_{anc}_BURDEN.SKAT-O.tsv"

        if os.path.isfile(path_skat) and os.path.getsize(path_skat) > 0:
            try:
                df = pd.read_csv(path_skat, sep='\t')
                if not df.empty:
                    all_skat.append(df)
            except Exception as e:
                print(f"  warn SKAT {ds} {anc}: {e}")

        if os.path.isfile(path_skato) and os.path.getsize(path_skato) > 0:
            try:
                df = pd.read_csv(path_skato, sep='\t')
                if not df.empty:
                    all_skato.append(df)
            except Exception as e:
                print(f"  warn SKAT-O {ds} {anc}: {e}")

ALL_SKAT  = pd.concat(all_skat,  ignore_index=True) if all_skat  else pd.DataFrame()
ALL_SKATO = pd.concat(all_skato, ignore_index=True) if all_skato else pd.DataFrame()

if 'KERNEL' not in ALL_SKAT.columns:
    ALL_SKAT.insert(4,  'KERNEL', 'SKAT')
if 'KERNEL' not in ALL_SKATO.columns:
    ALL_SKATO.insert(4, 'KERNEL', 'SKAT-O')

print(f"  SKAT total:   {len(ALL_SKAT):,} rows")
print(f"  SKAT-O total: {len(ALL_SKATO):,} rows")


def add_corrections(df, pval_col='Pvalue'):
    """FDR-BH + Bonferroni per group (ANCESTRY x DATASET x CLASS).

    rvtest can output negative p-values (numeric overflow in the chi-square
    statistic) or an exact 0 (underflow, for very strong associations).
    Negative values are clipped to 0; exact zeros are excluded from
    multipletests and assigned FDR=0 / Bonferroni=0 directly, since those
    are the most significant hits in the analysis.
    """
    if df.empty or pval_col not in df.columns:
        return df
    df = df.copy()
    df[pval_col] = df[pval_col].astype(float)
    n_neg = (df[pval_col] < 0).sum()
    if n_neg > 0:
        print(f"  {n_neg} negative p-values clipped to 0")
    df[pval_col] = df[pval_col].clip(lower=0.0)

    df['FDR_BH']     = np.nan
    df['Bonferroni'] = np.nan
    for (anc, ds, cls), grp in df.groupby(['ANCESTRY', 'DATASET', 'CLASS']):
        mask_valid = grp[pval_col].notna() & (grp[pval_col] > 0)
        idx_valid = grp[mask_valid].index
        if len(idx_valid) == 0:
            continue
        pvals = grp.loc[idx_valid, pval_col].values
        _, q_bh, _, _   = multipletests(pvals, method='fdr_bh')
        _, p_bonf, _, _ = multipletests(pvals, method='bonferroni')
        df.loc[idx_valid, 'FDR_BH']     = q_bh
        df.loc[idx_valid, 'Bonferroni'] = p_bonf
        idx_zero = grp[grp[pval_col] == 0].index
        df.loc[idx_zero, 'FDR_BH']     = 0.0
        df.loc[idx_zero, 'Bonferroni'] = 0.0
    return df


print(f"\n{datetime.now().strftime('%H:%M:%S')}  Applying FDR-BH and Bonferroni...")
ALL_SKAT  = add_corrections(ALL_SKAT)
ALL_SKATO = add_corrections(ALL_SKATO)

ALL_COMBINED = pd.concat([ALL_SKAT, ALL_SKATO], ignore_index=True)
out_global = f"{DIR_RESU}/ALL_BURDEN_FDR.tsv"
ALL_COMBINED.to_csv(out_global, sep='\t', index=False)
print(f"\nGlobal table saved: {out_global}")
print(f"Total rows: {len(ALL_COMBINED):,}")

print("\n=== Hits by significance threshold ===\n")
for kernel in ['SKAT', 'SKAT-O']:
    sub = ALL_COMBINED[ALL_COMBINED['KERNEL'] == kernel]
    if sub.empty:
        continue
    n_bonf_per = (sub['Pvalue'] < BONF_PER_ANC).sum()
    n_fdr_05   = (sub['FDR_BH'] < 0.05).sum()
    n_fdr_10   = (sub['FDR_BH'] < 0.10).sum()
    n_sugg     = (sub['Pvalue'] < 1e-4).sum()
    n_zero     = (sub['Pvalue'] == 0).sum()
    print(f"--- {kernel} ---")
    print(f"  Bonferroni per-ancestry (alpha={BONF_PER_ANC:.2e}): {n_bonf_per}")
    print(f"  FDR-BH < 0.05:                                 {n_fdr_05}")
    print(f"  FDR-BH < 0.10:                                 {n_fdr_10}")
    print(f"  Suggestive (P < 1e-4):                         {n_sugg}")
    print(f"  P = 0 (underflow):                             {n_zero}")
    print()

## Preliminary results summary


In [ ]:
import pandas as pd

DIR_RESU = "/home/jupyter/workspace/ws_files/Novalis_v3_R12/Results_union"
ALPHA_BONF = 0.05 / 5915
ORDER = ['AAC','AFR','AJ','AMR','CAS','EAS','EUR','FIN','MDE','SAS','CAH']
N_SAMPLES = {
    'AAC': 17785, 'AFR': 19397, 'AJ': 13774, 'AMR': 15804, 'CAS': 15111,
    'EAS': 15331, 'EUR': 20022, 'FIN': 2287,  'MDE': 15780, 'SAS': 12876, 'CAH': 17449
}

df = pd.read_csv(f"{DIR_RESU}/ALL_BURDEN_FDR.tsv", sep='\t')
print(f"ALL_BURDEN_FDR.tsv: {len(df):,} rows | KERNEL: {list(df['KERNEL'].unique())} | DATASET: {list(df['DATASET'].unique())}\n")

# A. Save FDR < 0.05 hits, split by dataset
for ds in ['NBA', 'WGS']:
    hits  = df[(df['FDR_BH'] < 0.05) & (df['DATASET'] == ds)].copy()
    out   = f"{DIR_RESU}/FDR_HITS_for_annotation_{ds}.tsv"
    hits.to_csv(out, sep='\t', index=False)
    skat  = len(hits[hits['KERNEL'] == 'SKAT'])
    skato = len(hits[hits['KERNEL'] == 'SKAT-O'])
    print(f"FDR_HITS_for_annotation_{ds}.tsv -> {len(hits):,} rows  (SKAT: {skat} | SKAT-O: {skato})")

print()

# B. Summary table per ancestry and dataset
def make_summary(df_ds):
    rows = []
    for anc in ORDER:
        sub = df_ds[df_ds['ANCESTRY'] == anc]
        if len(sub) == 0:
            rows.append({'ANC': anc, 'N': f"{N_SAMPLES[anc]:,}",
                         'Bonf_SKAT': '-', 'Bonf_SKATO': '-',
                         'FDR_SKAT': '-', 'FDR_SKATO': '-',
                         'Best_p': 'n/a', 'Best_HAR': 'no data'})
            continue
        bonf = sub[sub['Bonferroni'] < ALPHA_BONF]
        fdr  = sub[sub['FDR_BH'] < 0.05]
        top  = sub.loc[sub['Pvalue'].idxmin()]
        rows.append({
            'ANC'        : anc,
            'N'          : f"{N_SAMPLES[anc]:,}",
            'Bonf_SKAT'  : len(bonf[bonf['KERNEL'] == 'SKAT']),
            'Bonf_SKATO' : len(bonf[bonf['KERNEL'] == 'SKAT-O']),
            'FDR_SKAT'   : len(fdr[fdr['KERNEL'] == 'SKAT']),
            'FDR_SKATO'  : len(fdr[fdr['KERNEL'] == 'SKAT-O']),
            'Best_p'     : f"{top['Pvalue']:.2e}" if top['Pvalue'] > 0 else "~0 (underflow)",
            'Best_HAR'   : (top['GENE'][:40] + '...') if len(top['GENE']) > 40 else top['GENE'],
        })
    return pd.DataFrame(rows)

for ds in ['NBA', 'WGS']:
    print(f"{'=' * 110}")
    print(f"  SUMMARY - {ds}")
    print(f"{'=' * 110}")
    display(make_summary(df[df['DATASET'] == ds]))
    print()

# C. Top 10 unique HARs per kernel x dataset
for kernel in ['SKAT', 'SKAT-O']:
    for ds in ['NBA', 'WGS']:
        sub = df[(df['KERNEL'] == kernel) & (df['DATASET'] == ds)]
        if len(sub) == 0:
            print(f"No data: {kernel} - {ds}\n")
            continue
        top10 = (
            sub
            .sort_values('Pvalue')
            .drop_duplicates(subset=['ANCESTRY', 'GENE'])
            [['ANCESTRY', 'GENE', 'CLASS', 'Pvalue', 'FDR_BH', 'Bonferroni']]
            .head(10)
            .reset_index(drop=True)
        )
        print(f"{'=' * 110}")
        print(f"  TOP 10 UNIQUE HARs - {kernel} - {ds}")
        print(f"{'=' * 110}")
        display(top10)
        print()

In [ ]:
# Top hits per ancestry (FDR-BH < 0.05 AND Bonferroni per-ancestry)
# Self-contained - only needs ALL_BURDEN_FDR.tsv on disk
import os
import pandas as pd

DIR_RESU     = "/home/jupyter/workspace/ws_files/Novalis_v3_R12/Results_union"
GLOBAL_TSV   = f"{DIR_RESU}/ALL_BURDEN_FDR.tsv"
BONF_PER_ANC = 0.05 / 5915   # 8.45e-6
FDR_THRESH   = 0.05
# Some ancestries are underpowered (small N and/or extreme case-control imbalance): still run,
# but not treated as standalone findings here - reserved for a future joint meta-analysis.
LOW_POWER_ANC = {"FIN", "MDE", "SAS"}
TOP_N        = 5

# Load only the needed columns
_cols_avail  = pd.read_csv(GLOBAL_TSV, sep='\t', nrows=0).columns.tolist()
COLS_USE     = [c for c in ["ANCESTRY", "DATASET", "GENE", "CLASS", "KERNEL",
                             "Pvalue", "FDR_BH", "NumVar", "N_INFORMATIVE"]
                if c in _cols_avail]

df = pd.read_csv(GLOBAL_TSV, sep='\t', usecols=COLS_USE)
df = df[~df["ANCESTRY"].isin(LOW_POWER_ANC)].copy()
df["Pvalue"] = pd.to_numeric(df["Pvalue"], errors="coerce")
df["FDR_BH"] = pd.to_numeric(df["FDR_BH"], errors="coerce")

# Only hits passing both thresholds
hits = df[(df["Pvalue"] < BONF_PER_ANC) & (df["FDR_BH"] < FDR_THRESH)].copy()
hits = hits.sort_values(["ANCESTRY", "KERNEL", "Pvalue"])

SHOW = [c for c in ["DATASET", "GENE", "CLASS", "KERNEL",
                     "Pvalue", "FDR_BH", "NumVar", "N_INFORMATIVE"] if c in hits.columns]

pd.set_option("display.max_colwidth", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.2e}".format)

ancestries_hit = sorted(hits["ANCESTRY"].unique())
all_anc        = sorted(df["ANCESTRY"].unique())

print(f"Bonferroni per-ancestry alpha = {BONF_PER_ANC:.2e}  |  FDR-BH < {FDR_THRESH}")
print(f"Not reported as standalone findings (underpowered): {', '.join(sorted(LOW_POWER_ANC))}")
print()

if not ancestries_hit:
    print("No ancestry passed both thresholds.")
else:
    for anc in ancestries_hit:
        sub = hits[hits["ANCESTRY"] == anc]
        print("=" * 90)
        print(f"  {anc}  ({len(sub)} hits)")
        print("=" * 90)
        for kernel in ["SKAT-O", "SKAT"]:
            k_sub = sub[sub["KERNEL"] == kernel].head(TOP_N)
            if k_sub.empty:
                continue
            print(f"  [{kernel}]")
            print(k_sub[SHOW].to_string(index=False))
            print()

    no_hits = [a for a in all_anc if a not in ancestries_hit]
    if no_hits:
        print("-" * 50)
        print(f"No significant hits: {', '.join(no_hits)}")